In [30]:
import pandas as pd
import psycopg2

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    dbname="quickmart",
    user="postgres",
    password="postgres"
)

In [7]:
customers = pd.read_csv('data/customers.csv')
products = pd.read_csv('data/products.csv')
sales = pd.read_csv('data/sales.csv')
stores = pd.read_csv('data/stores.csv')

In [8]:
customers.head()

,customer_id,customer_name,email,city,state,customer_segment
0,C000001,David Okoro,customer000001@quickmart.example,Ikeja,Lagos,Mass Market
1,C000002,Peter Garba,customer000002@quickmart.example,Maiduguri,Borno,Mass Market
2,C000003,Hauwa Ogunleye,customer000003@quickmart.example,Port Harcourt,Rivers,Mass Market
3,C000004,Chinedu Okafor,customer000004@quickmart.example,Maiduguri,Borno,Mass Market
4,C000005,Chinedu Musa,customer000005@quickmart.example,Jos,Plateau,Mass Market


In [9]:
# Start by creating the date dimension

sales['sale_date'] = pd.to_datetime(sales['sale_date'])

dim_date = pd.DataFrame({
    "full_date": sales['sale_date'].drop_duplicates().sort_values()}
)

dim_date['date_key'] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quater'] = dim_date["full_date"].dt.quarter

In [10]:
# Generate surrogate keys

dim_customer = customers.copy()

dim_customer['customer_key'] = range(1, len(dim_customer) + 1)

In [11]:
# Generate surrogate keys

dim_products = products.copy()

dim_products['product_key'] = range(1, len(dim_products) + 1)

In [12]:
dim_stores = stores.copy()

dim_stores['store_key'] = range(1, len(dim_stores) + 1)

In [13]:
dim_sales = sales.copy()

dim_sales['sale_key'] = range(1, len(dim_sales) + 1)

In [14]:
# Fact Tables

fact_sales = sales.copy()

fact_sales= fact_sales.merge(
    dim_customer[['customer_id', 'customer_key']],
    on='customer_id',
    how='left'
)

fact_sales= fact_sales.merge(
    dim_products[['product_id', 'product_key']],
    on='product_id',
    how='left'
)

fact_sales= fact_sales.merge(
    dim_stores[['store_id', 'store_key']],
    on='store_id',
    how='left'
)

fact_sales= fact_sales.merge(
    dim_date[['full_date', 'date_key']],
    left_on='sale_date',
    right_on='full_date',
    how='left'
)




In [16]:
fact_sales.head()

,sale_id,sale_date,customer_id,product_id,store_id,quantity,unit_price,discount_pct,payment_method,customer_key,product_key,store_key,full_date,date_key
0,1,2026-03-24,C003576,P00189,S017,2,87000,0,Bank Transfer,3576,189,17,2026-03-24,20260324
1,2,2025-11-11,C008570,P00373,S029,1,175000,0,Cash,8570,373,29,2025-11-11,20251111
2,3,2026-01-15,C004439,P00232,S033,2,126000,10,Mobile Money,4439,232,33,2026-01-15,20260115
3,4,2026-07-05,C007041,P00399,S046,1,577000,0,Bank Transfer,7041,399,46,2026-07-05,20260705
4,5,2026-02-26,C007482,P00097,S002,1,204000,15,Card,7482,97,2,2026-02-26,20260226


In [17]:
fact_sales['sale_amount'] = fact_sales['quantity'] * fact_sales['unit_price']

In [18]:
# Select final columns

fact_sales = fact_sales[[
    'sale_id','date_key','customer_key','product_key','store_key','quantity','unit_price','sale_amount'
]]

In [22]:
fact_sales.head()

,sale_id,date_key,customer_key,product_key,store_key,quantity,unit_price,sale_amount
0,1,20260324,3576,189,17,2,87000,174000
1,2,20251111,8570,373,29,1,175000,175000
2,3,20260115,4439,232,33,2,126000,252000
3,4,20260705,7041,399,46,1,577000,577000
4,5,20260226,7482,97,2,1,204000,204000


In [ ]:
# Write data into data warehouse (Postgres)
conn = psycopg2.connect(database = "datacamp_courses",
                        user = "datacamp",
                        host= 'localhost',
                        password = "postgresql_tutorial",
                        port = 5432)